In [ ]:
import torch
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Добавляем корень проекта в путь
ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from osc_tools.ml.models import ConvKAN
from osc_tools.visualization.kan_plot import plot_kan_activation
from osc_tools.ml.config import ModelConfig

EXPERIMENTS_DIR = ROOT_DIR / 'experiments' / 'kan_activations'
print(f"Experiments Dir: {EXPERIMENTS_DIR}")

In [ ]:
def load_model(activation_name, num_classes=89, in_channels=12):
    model_path = EXPERIMENTS_DIR / f"ConvKAN_{activation_name}.pt"
    if not model_path.exists():
        print(f"Model not found: {model_path}")
        return None
    
    # Параметры должны совпадать с теми, что были при обучении
    # В run_kan_activations.py: base_filters=8, grid_size=5
    params = {
        "in_channels": in_channels,
        "num_classes": num_classes,
        "base_filters": 8,
        "grid_size": 5,
        "base_activation": getattr(torch.nn, activation_name) if hasattr(torch.nn, activation_name) else torch.nn.SiLU
    }
    
    model = ConvKAN(**params)
    state_dict = torch.load(model_path, map_location='cpu')
    model.load_state_dict(state_dict)
    model.eval()
    return model

In [ ]:
# Загрузка модели с SiLU (как основной)
model_silu = load_model("SiLU")

## Визуализация первого слоя (Convolutional KAN Layer)

ConvKAN состоит из сверточных слоев, где ядро свертки параметризуется сплайнами. 
В `ConvKAN` (osc_tools/ml/models.py) первый слой это `KANConv1d`.
Внутри `KANConv1d` веса хранятся специфично. Нам нужно посмотреть реализацию `KANConv1d` чтобы понять как достать `KANLinear` слои или как визуализировать ядра.

In [ ]:
# Давайте посмотрим структуру модели
if model_silu:
    print(model_silu)